# Import Modules

In [1]:
import importlib
import os
import sys

import joblib
import numpy as np
import pandas as pd
import polars as pl
import seaborn as sns
import sklearn.metrics as skm

os.chdir("../")
sys.path.insert(0, os.getcwd())

In [2]:
from morai import models
from morai.dashboard.utils import dashboard_helper as dh
from morai.experience import charters, credibility, eda, tables
from morai.forecast import metrics, preprocessors
from morai.utils import custom_logger, helpers

In [3]:
logger = custom_logger.setup_logging(__name__)

In [4]:
# update log level if wanting more logging
custom_logger.set_log_level("INFO")

In [5]:
pd.options.display.float_format = "{:,.2f}".format

In [6]:
# default is "plotly_mimetype+notebook", however that takes up space.
# "plotly_mimetype+notebook_connected" seems to save space
import plotly.io as pio

pio.renderers.default = "plotly_mimetype+notebook_connected"

# Load Data

In [7]:
pl_parquet_path = helpers.FILES_PATH / "dataset" / "model_data.parquet"

In [8]:
# reading in the dataset
# `enable_string_cache` helps with categorical type values
pl.enable_string_cache()
lzdf = pl.scan_parquet(
    pl_parquet_path,
)

In [9]:
initial_row_count = lzdf.select(pl.len()).collect().item()
print(
    f"row count: {initial_row_count:,} \n"
    f"exposures: {lzdf.select([pl.col('amount_exposed').sum()]).collect()[0,0]:,}"
)

row count: 1,096,727 
exposures: 4,284,836,591,111.3477


In [10]:
grouped_df = lzdf.collect()

In [11]:
grouped_df = grouped_df.to_pandas()

# Credibility

## Limited Fluctuation

In [27]:
qx = 0.5
n = 1
exposure = pd.DataFrame({"rate": [n]})
credibility.limited_fluctuation(
    df=exposure, measure="rate", sd=(n * qx * (1 - qx)) ** (1 / 2), u=n * qx
)

 2025-06-19 01:01:31 | morai.experience.credibility | INFO     | Credibility calculated using 'limited fluctuation' on 'rate'.
Dataframe does not need to be seriatim.
Created column 'credibility_lf'.
Full credibility threshold: 541.1
Probability measure within range: 0.9
Range +/-: 0.1
Standard deviation: 0.5
Mean: 0.5 


,rate,credibility_lf
0,1,0.04


In [12]:
charters.chart(
    df=credibility.limited_fluctuation(
        df=grouped_df, measure="death_count", groupby_cols=["attained_age"]
    ),
    x_axis="attained_age",
    y_axis="credibility_lf",
    type="line",
)

 2025-06-05 22:50:28 | morai.experience.credibility | INFO     | Credibility calculated using 'limited fluctuation' on 'death_count'.
Dataframe does not need to be seriatim.
Created column 'credibility_lf'.
Full credibility threshold: 1,082.2
Probability measure within range: 0.9
Range +/-: 0.1
Standard deviation: 1.0
Mean: 1.0 


## Asymptotic

In [13]:
charters.chart(
    df=credibility.asymptotic(
        df=grouped_df, measure="policies_exposed", groupby_cols=["attained_age"], k=5000
    ),
    x_axis="attained_age",
    y_axis="credibility_as",
    type="line",
)

 2025-06-05 22:50:33 | morai.experience.credibility | INFO     | Credibility calculated using 'asymptotic' on 'policies_exposed'.
Dataframe does not need to be seriatim.
Created column 'credibility_as'.
Constant k: 5000. 


In [14]:
charters.chart(
    df=credibility.asymptotic(
        df=grouped_df,
        measure="policies_exposed",
        groupby_cols=["attained_age", "duration"],
        k=5000,
    ),
    x_axis="attained_age",
    color="credibility_as",
    y_axis="duration",
    type="heatmap",
)

 2025-06-05 22:50:33 | morai.experience.credibility | INFO     | Credibility calculated using 'asymptotic' on 'policies_exposed'.
Dataframe does not need to be seriatim.
Created column 'credibility_as'.
Constant k: 5000. 


## VM20 Buhlmann

In [15]:
charters.chart(
    df=credibility.vm20_buhlmann_approx(
        df=grouped_df,
        a_col="exp_amt_vbt15",
        b_col="cen2momp1wmi_byamt",
        c_col="cen2momp2wmi_byamt",
        groupby_cols=["attained_age"],
    ),
    x_axis="attained_age",
    y_axis="credibility_vm20_approx",
    type="line",
)

 2025-06-05 22:50:33 | morai.experience.credibility | INFO     | Credibility calculated using 'SOA VM-20 approximation'.
Created column 'credibility_vm20_approx'.
Dataframe does not need to be seriatim.
 


# Relative Risk Table Fit

In [16]:
rates = [
    "vbt15_rr50",
    "vbt15_rr60",
    "vbt15_rr70",
    "vbt15_rr80",
    "vbt15_rr90",
    "vbt15",
    "vbt15_rr110",
]
for rate in rates:
    grouped_df = tables.map_rates(
        df=grouped_df,
        rate=rate,
        rate_to_df_map={
            "attained_age": "attained_age",
            "smoker_status": "smoker_status",
            "sex": "sex",
            "duration": "duration",
        },
    )

 2025-06-05 22:50:38 | morai.experience.tables | INFO     | mapping rate: 'qx_vbt15_rr50' with format: 'soa' 
 2025-06-05 22:50:39 | morai.experience.tables | INFO     | the mapped rates are based on the following keys: ['attained_age', 'smoker_status', 'sex', 'duration'] 
 2025-06-05 22:50:42 | morai.experience.tables | INFO     | mapping rate: 'qx_vbt15_rr60' with format: 'soa' 
 2025-06-05 22:50:44 | morai.experience.tables | INFO     | the mapped rates are based on the following keys: ['attained_age', 'smoker_status', 'sex', 'duration'] 
 2025-06-05 22:50:47 | morai.experience.tables | INFO     | mapping rate: 'qx_vbt15_rr70' with format: 'soa' 
 2025-06-05 22:50:48 | morai.experience.tables | INFO     | the mapped rates are based on the following keys: ['attained_age', 'smoker_status', 'sex', 'duration'] 
 2025-06-05 22:50:51 | morai.experience.tables | INFO     | mapping rate: 'qx_vbt15_rr80' with format: 'soa' 
 2025-06-05 22:50:53 | morai.experience.tables | INFO     | the mapp

In [17]:
result = []
filtered_df = grouped_df[grouped_df["class_enh"] == "4_4"]
for rate in rates:
    score = metrics.calculate_metrics(
        y_true=filtered_df["death_claim_amount"],
        y_pred=filtered_df["amount_exposed"] * filtered_df[f"qx_{rate}"],
        metrics=["r2_score", "smape", "mean_absolute_error"],
    )
    score["rate"] = rate
    result.append(score)
result = pd.DataFrame(result)
result = result.sort_values(by="_r2_score", ascending=False)
result

,_r2_score,_smape,_mean_absolute_error,rate
6,0.04,1.98,"22,771.22",vbt15_rr110
5,0.04,1.98,"22,026.42",vbt15
4,0.04,1.98,"21,188.98",vbt15_rr90
3,0.04,1.98,"20,529.29",vbt15_rr80
2,0.04,1.98,"19,874.13",vbt15_rr70
1,0.03,1.99,"19,109.01",vbt15_rr60
0,0.03,1.99,"18,344.75",vbt15_rr50


In [18]:
charters.compare_rates(
    df=filtered_df,
    x_axis="attained_age",
    rates=["qx_raw", "qx_vbt15", "qx_vbt15_rr110"],
    weights=["amount_exposed"],
    y_log=True,
    # x_bins=6,
    display=True,
)

 2025-06-05 22:51:06 | morai.experience.charters | INFO     | The weights list is 1 long and should be 3 long. Using the first weight for all weights. 


# Importance

As seen in the feature selection, the most important variables from permutation were below:
- attained age
- duration
- observation year (not statistically significant however)
- class
- faceband

For credibility calculations these will be split.

In [36]:
model_name = "glm"

GLM = models.core.GLM()
GLM.model = joblib.load(f"files/models/{model_name}.joblib")
logger.info(f"loaded model '{model_name}'. type: {type(GLM.model)}")

 2025-06-05 23:25:14 | __main__ | INFO     | loaded model 'glm'. type: <class 'statsmodels.genmod.generalized_linear_model.GLMResultsWrapper'> 


In [30]:
odds = GLM.get_odds()

In [22]:
pivot_list = []
for feature in feature_dict["ohe"]:
    pivot_df = grouped_df.pivot_table(
        values="death_count", index=feature, aggfunc="sum", observed=False
    ).reset_index()
    pivot_df["column"] = feature
    pivot_df = pivot_df.rename(columns={feature: "value"})
    pivot_df = pivot_df[["column", "value", "death_count"]]
    pivot_df["odds"] = pivot_df.apply(
        lambda row: odds.get(f"{feature}_{row['value']}", None), axis=1
    )
    pivot_list.append(pivot_df)
result_df = pd.concat(pivot_list, ignore_index=True)
result_df.sort_values(by="odds")

,column,value,death_count,odds
4,binned_face,"05: 5,000,000+",330,0.64
20,class_enh,4_1,2263,0.70
1,binned_face,"04: 250,000 - 4,999,999",17558,0.76
2,binned_face,"03: 100,000 - 249,999",29887,0.80
15,class_enh,4_2,2200,0.86
0,binned_face,"02: 25,000 - 99,999",55862,0.87
18,class_enh,3_2,3257,0.88
21,class_enh,3_1,2121,0.88
8,insurance_plan,ULSG,9662,1.04
11,class_enh,U_U,1043,1.06


In [34]:
# The more that a feature increases a deviance when it is removed the more impactful the feature is to the model
# and would be considered an important feature.
base_features = [f for f in X.columns if f not in ["sex", "smoker_status"]]
GLM.get_feature_contributions(X, y, weights=weights, base_features=base_features)

,features,contribution,deviance
0,all,1.00,"92,367,000,240.03"
1,base,0.00,"93,651,582,303.26"
2,sex,0.24,"92,679,606,423.62"
3,smoker_status,0.76,"93,346,000,843.96"


# Reload

In [14]:
importlib.reload(tables)

<module 'morai.experience.tables' from 'C:\\Users\\johnk\\Desktop\\github\\morai\\morai\\experience\\tables.py'>

In [37]:
importlib.reload(metrics)

<module 'morai.forecast.metrics' from 'C:\\Users\\johnk\\Desktop\\github\\morai\\morai\\forecast\\metrics.py'>

# Utilities

In [25]:
helpers.memory_usage_jupyter().head(10)

,object,size_mb
0,grouped_df,411.26
1,X,49.18
2,filtered_df,19.08
3,weights,8.37
4,y,8.37
5,result_df,0.00
6,odds,0.00
7,pivot_df,0.00
8,result,0.00
9,preprocess_dict,0.00
